# Xiangqi AI — Silver → Gold (Policy + Offline Q-Learning)

Reads **only two explicitly selected Silver full runs**: CCPD and Alpha-Beta. Stages copies under /content, verifies SHA-256, runs the backend CLI, validates all output digests, then publishes a new immutable Gold run on Google Drive. This notebook does not implement AI training or rewrite Silver. It builds a **new immutable Gold run** with configurable potential-based reward shaping. Existing Gold v1 remains unchanged.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
SILVER_ROOT = DRIVE_PROJECT / "data" / "02_silver"
GOLD_ROOT = DRIVE_PROJECT / "data" / "03_gold"
SOURCE_REPO = Path("/content/ML-Advance-Group")  # Existing clone, never switch it.

# Chọn chính xác phiên bản full, KHÔNG chọn sample hoặc validated76.
SILVER_RUNS = [
    "ccpd_full_20261107T080132614436Z",
    "alphabeta_full_20261107T074509139310Z",
]
GIT_BRANCH = "main"
SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

# Versioned RL reward; same gamma must be used when training DQN.
REWARD_MODE = "shaped"         # "sparse" exactly reproduces Gold v1 reward
GAMMA = 0.99
SHAPING_WEIGHT = 0.15
MATERIAL_SCALE = 300.0
PAWN_ADVANCEMENT_WEIGHT = 1.0

In [ ]:
import subprocess
import tempfile
import sys

# Do not run "git switch" in the shared Silver checkout.
# A detached Git worktree reuses its Git objects without touching dirty files.
workspace = Path(tempfile.mkdtemp(prefix="xiangqi_gold_", dir="/content"))
REPO_DIR = workspace / "ML-Advance-Group"
BACKEND_DIR = REPO_DIR / "project" / "backend"

if SOURCE_REPO.is_dir() and (SOURCE_REPO / ".git").exists():
    subprocess.run(
        ["git", "-C", str(SOURCE_REPO), "fetch", "origin", GIT_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(SOURCE_REPO), "worktree", "add", "--detach",
         str(REPO_DIR), "FETCH_HEAD"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
         "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)],
    check=True,
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if not (BACKEND_DIR / "scripts" / "build_gold.py").is_file():
    raise FileNotFoundError("Gold CLI not found: scripts/build_gold.py")
print("Backend Gold revision:", REVISION)
print("Backend:", BACKEND_DIR)

In [ ]:
import hashlib
import shutil
import tempfile
from datetime import datetime, timezone

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Always create a fresh local workspace: deleting Drive files does NOT remove
# previously staged files under /content, even when the same cell is rerun.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_NAME = f"gold_{REWARD_MODE}_{stamp}"
STAGING_ROOT = Path(
    tempfile.mkdtemp(prefix="xiangqi_gold_stage_", dir="/content")
)
local_inputs = []

for name in SILVER_RUNS:
    source = SILVER_ROOT / name
    target = STAGING_ROOT / name
    if not source.is_dir():
        raise FileNotFoundError(f"Silver run missing: {source}")
    target.mkdir(parents=True, exist_ok=False)
    for filename in ("games.jsonl", "transitions.jsonl"):
        src = source / filename
        dst = target / filename
        if not src.is_file():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        if sha256_file(src) != sha256_file(dst):
            raise IOError(f"SHA-256 mismatch while staging: {src}")
    local_inputs.append(target)
    print("Staged", name)

print("New local staging root:", STAGING_ROOT)
print("New Gold run name:", RUN_NAME)

In [ ]:
import json

LOCAL_OUTPUT = Path("/content/xiangqi_gold_runs") / RUN_NAME
command = [
    sys.executable, "-m", "scripts.build_gold",
    "--silver-dirs", *map(str, local_inputs),
    "--output-dir", str(LOCAL_OUTPUT),
    "--seed", str(SEED),
    "--train-ratio", str(TRAIN_RATIO),
    "--val-ratio", str(VAL_RATIO),
    "--reward-mode", REWARD_MODE,
    "--gamma", str(GAMMA),
    "--shaping-weight", str(SHAPING_WEIGHT),
    "--material-scale", str(MATERIAL_SCALE),
    "--pawn-advancement-weight", str(PAWN_ADVANCEMENT_WEIGHT),
]

if LOCAL_OUTPUT.exists():
    # Rerunning just this cell should verify an existing complete local run,
    # not overwrite or fail after expensive Gold generation.
    if not (LOCAL_OUTPUT / "report.json").is_file():
        raise FileExistsError(
            f"Incomplete local Gold output: {LOCAL_OUTPUT}. "
            "Rerun the staging cell to generate a fresh RUN_NAME."
        )
    print("Local Gold run already exists; validating checksums:", LOCAL_OUTPUT)
else:
    subprocess.run(command, cwd=BACKEND_DIR, check=True)

report = json.loads((LOCAL_OUTPUT / "report.json").read_text(encoding="utf-8"))
if report.get("reward_config", {}).get("mode") != REWARD_MODE:
    raise ValueError("Existing Gold run uses a different reward mode")
for filename, expected in report["output_sha256"].items():
    actual = sha256_file(LOCAL_OUTPUT / filename)
    if actual != expected:
        raise IOError(f"Corrupt Gold output {filename}")

assert sum(report["transitions_by_split"].values()) == report["transitions"]
assert sum(report["games_by_split"].values()) == report["accepted_games"]
print("Gold games:", report["accepted_games"], "| transitions:", report["transitions"])
print("By split:", report["transitions_by_split"])
print("Reward definition:", report["reward_definition"])
print("Reward configuration:", report["reward_config"])

In [ ]:
# Idempotent publishing: a prior successful upload should be verified,
# not repeated. Resume a previously interrupted .uploading directory.
GOLD_ROOT.mkdir(parents=True, exist_ok=True)
final_path = GOLD_ROOT / RUN_NAME
temp_path = GOLD_ROOT / f"{RUN_NAME}.uploading"
local_files = sorted(p for p in LOCAL_OUTPUT.iterdir() if p.is_file())
local_names = {p.name for p in local_files}

def verify_uploaded(folder):
    uploaded_names = {p.name for p in folder.iterdir() if p.is_file()}
    if uploaded_names != local_names:
        raise IOError(f"Gold output file mismatch in {folder}")
    for local_file in local_files:
        target = folder / local_file.name
        if sha256_file(local_file) != sha256_file(target):
            raise IOError(f"Gold output checksum mismatch: {target}")

if final_path.exists():
    verify_uploaded(final_path)
    print("Gold run already published and verified:", final_path)
else:
    if not temp_path.exists():
        temp_path.mkdir(parents=True, exist_ok=False)
    for local_file in local_files:
        target = temp_path / local_file.name
        if target.exists():
            if sha256_file(local_file) != sha256_file(target):
                raise IOError(
                    f"Conflicting partial upload: {target}. "
                    "Do not overwrite; rerun staging for a fresh RUN_NAME."
                )
        else:
            shutil.copy2(local_file, target)
        if sha256_file(local_file) != sha256_file(target):
            raise IOError(f"Failed Drive upload checksum: {target}")
    verify_uploaded(temp_path)
    temp_path.rename(final_path)
    print("Published:", final_path)

print("Total transitions:", report["transitions"])
print("Git SHA:", REVISION)